In [1]:
from func import *

# Read data

In [6]:
train_df = pd.read_csv("train_df_clean.csv")
test_df = pd.read_csv("test_df_clean.csv")

train_df = train_df.drop(["HighSchoolDistrict_target_mean", "Flooring_target_mean", "LivingArea_std", "LotSizeSquareFeet_std", "AssociationFee_std"], axis=1)
no_missing_train_df = train_df.dropna(axis=1)

test_df = test_df.drop(["HighSchoolDistrict_target_mean", "Flooring_target_mean", "LivingArea_std", "LotSizeSquareFeet_std", "AssociationFee_std"], axis=1)
no_missing_test_df = test_df.dropna(axis=1)

train_df["log_ClosePrice"] = np.log1p(train_df["ClosePrice"])
train_df.drop(columns=["ClosePrice"], inplace=True)
test_df["log_ClosePrice"] = np.log1p(test_df["ClosePrice"])
test_df.drop(columns=["ClosePrice"], inplace=True)

train_df = no_missing_train_df
test_df = no_missing_test_df

In [7]:
train_df = train_df.rename(columns={"log_ClosePrice": "logClosePrice"})
test_df = test_df.rename(columns={"log_ClosePrice": "logClosePrice"})

# Elastic Net

In [4]:
from sklearn.linear_model import ElasticNet

In [8]:

# Tuning
en_param_grid = param_grid = {
    "model__alpha": np.logspace(-4, 2, 25),
    "model__l1_ratio": np.linspace(0.05, 0.95, 19),
}

res = grid_tune_with_make_model_pipeline(
    train_df=train_df,
    target_col="logClosePrice",
    model=ElasticNet(max_iter=20000),
    param_grid=param_grid,
    col_drop_list=["ClosePrice"],
    card_threshold=20,
    scoring="r2",
    cv=5
)

print(res["best_score"], res["best_params"])

Fitting 5 folds for each of 475 candidates, totalling 2375 fits
0.8628536196065241 {'model__alpha': 0.01778279410038923, 'model__l1_ratio': 0.05}


In [9]:
alpha = 0.01778279410038923
l_1_ratio = 0.05
model = ElasticNet(alpha=alpha,
                   l1_ratio=l_1_ratio)

elastic_net_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20
)
el_metrics = elastic_net_result["Metrics"]
print(el_metrics)

{'R2(log)': 0.8607197316796089, 'R2': 0.7554782907265002, 'MAPE': 17.464867232504787, 'MdAPE': 12.220412116358057, 'RMSE': 457331.8219661648, 'MAE': 212945.33180124752, 'Bias(mean residual)': -26235.445359768622, 'APE_95pct': 49.50770971536067, 'APE_99pct': 90.16710414349195, 'APE_max': 247.5146005851534, 'Train_R2(log)': 0.887148544277048, 'Test_R2(log)': 0.8607197316796089, 'R2_gap': 0.026428812597439122}


# Random Forest

In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

In [11]:
X_train = train_df.drop(columns=["logClosePrice"], axis=1)

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

nunique = X_train[cat_cols].nunique(dropna=False)

low_card_cols = nunique[(nunique <= 20)].index.tolist()
high_card_cols = nunique[(nunique >= 20)].index.tolist()


In [20]:

rf_param_dist = {
    "model__n_estimators": randint(300, 1000),
    "model__max_depth": [None] + list(range(10, 60, 10)),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "model__min_samples_leaf": randint(1, 15),
    "model__min_samples_split": randint(2, 20)
}

model = make_model_pipeline(model=RandomForestRegressor(
    n_jobs=1,
    random_state=42),
    num_cols=num_cols,
    low_card_cols=low_card_cols,
    high_card_cols=high_card_cols
)


rs = RandomizedSearchCV(
    model,
    param_distributions=rf_param_dist,
    n_iter=40,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

rs.fit(train_df.drop("logClosePrice", axis=1), train_df["logClosePrice"])
print(rs.best_params_)

KeyboardInterrupt: 

In [12]:
n_estimators = 713
max_depth = 40
max_features = 'log2'
min_samples_leaf = 8
min_samples_split = 13

rf_model = RandomForestRegressor(n_estimators=n_estimators,
                                 max_depth=max_depth,
                                 max_features=max_features,
                                 min_samples_leaf=min_samples_leaf,
                                 min_samples_split=min_samples_split)

rf_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=rf_model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20
)

rf_metrics = rf_result["Metrics"]
print(rf_metrics)

{'R2(log)': 0.7359626181640566, 'R2': 0.4973295278446741, 'MAPE': 23.425110327413666, 'MdAPE': 17.93336535068987, 'RMSE': 655713.9108958194, 'MAE': 304616.822902939, 'Bias(mean residual)': -167053.41102332977, 'APE_95pct': 63.0850302349009, 'APE_99pct': 103.71014988662225, 'APE_max': 255.33818754757345, 'Train_R2(log)': 0.9721304119856582, 'Test_R2(log)': 0.7359626181640566, 'R2_gap': 0.23616779382160158}


# XGB

In [13]:
from xgboost import XGBRegressor
from scipy.stats import randint, uniform, loguniform

In [14]:

xgb_param_dist = {
    "model__max_depth": randint(3, 7),                    # 3–6
    "model__min_child_weight": randint(8, 31),           # 8–30
    "model__learning_rate": loguniform(0.01, 0.08),      # small LR
    "model__n_estimators": randint(400, 1400),
    "model__subsample": uniform(0.6, 0.3),               # 0.6–0.9
    "model__colsample_bytree": uniform(0.5, 0.4),        # 0.5–0.9
    "model__reg_lambda": loguniform(1.0, 50.0),          # strong L2
    "model__reg_alpha": loguniform(1e-4, 2.0),           # mild–moderate L1
}

model = make_model_numeric_only_pipeline(model=XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=1),
 num_cols=num_cols)

xgb_rs = RandomizedSearchCV(
    model,
    param_distributions=xgb_param_dist,
    n_iter=40,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

xgb_rs.fit(train_df.drop("logClosePrice", axis=1), train_df["logClosePrice"])
print(xgb_rs.best_params_)

{'model__colsample_bytree': 0.8887128330883842, 'model__learning_rate': 0.07399059420398583, 'model__max_depth': 4, 'model__min_child_weight': 14, 'model__n_estimators': 785, 'model__reg_alpha': 0.11574364959605028, 'model__reg_lambda': 1.7848234036017752, 'model__subsample': 0.8993221455146825}


In [18]:
xgb_model = XGBRegressor(n_estimators=785,
                         learning_rate=0.07399059420398583,
                         max_depth=4,
                         subsample=0.8993221455146825,
                         colsample_bytree=0.8887128330883842,
                         min_child_weight=14,
                         reg_lambda=1.7848234036017752,
                         reg_alpha=0.11574364959605028)


xgb_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=xgb_model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20,
    num_scaler=None,
    num_only=True
)

xgb_metrics = xgb_result["Metrics"]
print(xgb_metrics)

{'R2(log)': 0.9091223097049991, 'R2': 0.860241340221354, 'MAPE': 13.639511135099397, 'MdAPE': 9.509651829476539, 'RMSE': 345749.90829510096, 'MAE': 169026.25639527023, 'Bias(mean residual)': -4911.154149774787, 'APE_95pct': 40.11574224366324, 'APE_99pct': 69.14250312677011, 'APE_max': 194.42621621776053, 'Train_R2(log)': 0.925324132563527, 'Test_R2(log)': 0.9091223097049991, 'R2_gap': 0.016201822858527892}


# Summary

In [19]:
models_summary = {
    "Elastic Net": el_metrics,
    "Random Forest": rf_metrics,
    "XGBoost": xgb_metrics,
}

df = pd.DataFrame.from_dict(models_summary).transpose()
df

,R2(log),R2,MAPE,MdAPE,RMSE,MAE,Bias(mean residual),APE_95pct,APE_99pct,APE_max,Train_R2(log),Test_R2(log),R2_gap
Elastic Net,0.860720,0.755478,17.464867,12.220412,457331.821966,212945.331801,-26235.445360,49.507710,90.167104,247.514601,0.887149,0.860720,0.026429
Random Forest,0.735963,0.497330,23.425110,17.933365,655713.910896,304616.822903,-167053.411023,63.085030,103.710150,255.338188,0.972130,0.735963,0.236168
XGBoost,0.909122,0.860241,13.639511,9.509652,345749.908295,169026.256395,-4911.154150,40.115742,69.142503,194.426216,0.925324,0.909122,0.016202
